In [ ]:
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

urllib.request.urlretrieve(url, "/content/sms_spam.zip")

with zipfile.ZipFile("/content/sms_spam.zip", "r") as z:
    z.extractall("/content/sms_dataset")

print("Dataset downloaded and extracted successfully!")

Dataset downloaded and extracted successfully!


In [ ]:
import os

dataset_path = "/content/sms_dataset"

print("Files inside the dataset folder:")

for file in os.listdir(dataset_path):
    print(file)

Files inside the dataset folder:
SMSSpamCollection
readme


In [ ]:
import pandas as pd

file_path = "/content/sms_dataset/SMSSpamCollection"

df = pd.read_csv(
    file_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("Dataset loaded successfully!")
print(df.head())

Dataset loaded successfully!
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [ ]:
# Check the number of messages
print("Dataset shape:", df.shape)

# Check column information
print("\nDataset information:")
print(df.info())

# Count spam and ham messages
print("\nMessage distribution:")
print(df["label"].value_counts())

Dataset shape: (5572, 2)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None

Message distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


In [ ]:
# Convert text labels into numbers
df["label_num"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

# Display the first five rows
print(df.head())

# Check the numerical labels
print("\nLabel mapping:")
print(df[["label", "label_num"]].drop_duplicates())

  label                                            message  label_num
0   ham  Go until jurong point, crazy.. Available only ...          0
1   ham                      Ok lar... Joking wif u oni...          0
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...          1
3   ham  U dun say so early hor... U c already then say...          0
4   ham  Nah I don't think he goes to usf, he lives aro...          0

Label mapping:
  label  label_num
0   ham          0
2  spam          1


In [ ]:
# Separate messages and labels

emails = df["message"]

labels = df["label_num"]

# Display the first five messages and labels

print("Messages (X):")
print(emails.head())

print("\nLabels (y):")
print(labels.head())

Messages (X):
0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: message, dtype: object

Labels (y):
0    0
1    0
2    1
3    0
4    0
Name: label_num, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    emails,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

print("Training labels:", len(y_train))
print("Testing labels:", len(y_test))

Training messages: 4457
Testing messages: 1115
Training labels: 4457
Testing labels: 1115


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
vec = TfidfVectorizer()

# Learn vocabulary from training data and transform it
X_train_tfidf = vec.fit_transform(X_train)

# Transform testing data using the same vocabulary
X_test_tfidf = vec.transform(X_test)

print("TF-IDF conversion completed!")

print("Training data shape:", X_train_tfidf.shape)
print("Testing data shape:", X_test_tfidf.shape)

TF-IDF conversion completed!
Training data shape: (4457, 7668)
Testing data shape: (1115, 7668)


In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Create the model
model = MultinomialNB()

# Train the model
model.fit(X_train_tfidf, y_train)

print("Spam detection model trained successfully!")

Spam detection model trained successfully!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Predict labels for testing messages
y_pred = model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy * 100, "%")

# Detailed performance report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Ham", "Spam"]
))

Model Accuracy: 96.05381165919282 %

Classification Report:
              precision    recall  f1-score   support

         Ham       0.96      1.00      0.98       966
        Spam       1.00      0.70      0.83       149

    accuracy                           0.96      1115
   macro avg       0.98      0.85      0.90      1115
weighted avg       0.96      0.96      0.96      1115



In [ ]:
print("========== SPAM DETECTOR RESULTS ==========")

print(f"Accuracy: {accuracy * 100:.2f}%")

print("Total testing messages:", len(y_test))

print("Correct predictions:", (y_test == y_pred).sum())

print("Incorrect predictions:", (y_test != y_pred).sum())

print("===========================================")

========== SPAM DETECTOR RESULTS ==========
Accuracy: 96.05%
Total testing messages: 1115
Correct predictions: 1071
Incorrect predictions: 44


In [ ]:
# Five challenging SMS messages

tests = [
    "Cl@im your fr33 g1ft card now",
    "Not free, just confirming lunch at 1pm",
    "Dear customer, congratulations! You have been selected to receive a cash reward. Reply now.",
    "Hey, are we still meeting tomorrow? I will bring the documents.",
    "URGENT! Your account has won a FREE prize. Click here to claim your reward."
]

# Convert messages into TF-IDF features
tests_tfidf = vec.transform(tests)

# Predict the messages
predictions = model.predict(tests_tfidf)

# Display results
for message, prediction in zip(tests, predictions):
    result = "SPAM" if prediction == 1 else "HAM"
    print("Message:", message)
    print("Prediction:", result)
    print("-" * 60)

Message: Cl@im your fr33 g1ft card now
Prediction: HAM
------------------------------------------------------------
Message: Not free, just confirming lunch at 1pm
Prediction: HAM
------------------------------------------------------------
Message: Dear customer, congratulations! You have been selected to receive a cash reward. Reply now.
Prediction: SPAM
------------------------------------------------------------
Message: Hey, are we still meeting tomorrow? I will bring the documents.
Prediction: HAM
------------------------------------------------------------
Message: URGENT! Your account has won a FREE prize. Click here to claim your reward.
Prediction: SPAM
------------------------------------------------------------


In [ ]:
# Legitimate messages containing suspicious words

extra_tests = [
    "I am not offering a prize, I just want to discuss our project.",
    "Please do not click any links, I am only sharing the meeting notes.",
    "My free time is available tomorrow, let us meet for lunch."
]

extra_tfidf = vec.transform(extra_tests)

extra_predictions = model.predict(extra_tfidf)

for message, prediction in zip(extra_tests, extra_predictions):
    result = "SPAM" if prediction == 1 else "HAM"
    print("Message:", message)
    print("Prediction:", result)
    print("-" * 60)

Message: I am not offering a prize, I just want to discuss our project.
Prediction: HAM
------------------------------------------------------------
Message: Please do not click any links, I am only sharing the meeting notes.
Prediction: HAM
------------------------------------------------------------
Message: My free time is available tomorrow, let us meet for lunch.
Prediction: HAM
------------------------------------------------------------


## Conclusion and What I Learned

This project demonstrates how Machine Learning can classify SMS messages as spam or ham using TF-IDF and Multinomial Naive Bayes.

### Key Results
- Model accuracy: **96.05%**
- Testing messages: **1,115**
- Correct predictions: **1,071**
- Incorrect predictions: **44**

### What I Learned
1. I learned how to preprocess text messages, convert text into numerical features using TF-IDF, and train a Multinomial Naive Bayes classifier.
2. I learned that machine learning models can achieve high accuracy but are not perfect. Unusual spelling, misleading words, and message context can cause incorrect predictions.

### Dataset
[UCI SMS Spam Collection Dataset](https://archive.ics.uci.edu/dataset/228/sms+spam+collection)